# Sentinel Gate - Stage 7b: family + signature metadata for the backtest (one short run)

**What this does**
1. Loads the code bundle and installs the *locked* dependencies (same `uv.lock` as the local sandbox).
2. Downloads the official EMBER2024 **Win64** feature-record zips (SHA-256 verified by `scripts/download_ember.py`).
3. Reads each raw record and keeps only: sha256, week (numbered exactly as Stage 1 cleaning), label, **family**, family_confidence and two Authenticode flags (num_certs, self_signed). **No feature vectors.**
4. Writes `sentinel_stage7b_meta_Win64.zip` to `/kaggle/working` (the notebook's Output).

EMBER2024 records carry no signer names (the Authenticode block is 8 numbers), so signature status is exported instead.

**Inputs to attach (Add Input):** your code-bundle dataset (`dist/sentinel_gate_code.zip`, Stage 7b version). No other input.

**Safety:** feature records only; no binaries are downloaded or executed; `EMBER2024-capa` is never touched.

**Kaggle settings:** Accelerator **None** (CPU only), Internet **On**. Then **Save Version -> Save & Run All (Commit)**. Expect roughly 15-30 minutes.


In [ ]:
# 1) Settings + platform + disk check
FILE_TYPE = "Win64"
REQUIRED_FREE_GB = 20

import os, shutil, subprocess, sys, zipfile, hashlib, glob, time, importlib.util, json
from pathlib import Path

ON_KAGGLE = ("KAGGLE_KERNEL_RUN_TYPE" in os.environ or "KAGGLE_URL_BASE" in os.environ
             or (Path("/kaggle/input").is_dir() and any(Path("/kaggle/input").iterdir())))
ON_COLAB = not ON_KAGGLE and ("COLAB_RELEASE_TAG" in os.environ
                              or importlib.util.find_spec("google.colab") is not None)
WHERE = "Kaggle notebook" if ON_KAGGLE else ("Google Colab" if ON_COLAB else "other notebook host")
candidates = [Path(p) for p in ["/content", "/kaggle/tmp", "/tmp"] if Path(p).exists()]
free = {p: shutil.disk_usage(p).free / 1e9 for p in candidates}
SCRATCH = max(free, key=free.get)
OUT_DIR = Path("/kaggle/working") if ON_KAGGLE else SCRATCH / "sentinel_results"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Host: {WHERE} | scratch: {SCRATCH} ({free[SCRATCH]:.1f} GB free) | outputs: {OUT_DIR}")
if free[SCRATCH] < REQUIRED_FREE_GB:
    raise SystemExit(f"Not enough disk: need {REQUIRED_FREE_GB} GB free. Tell Claude.")
CODE, DATA, VENV = SCRATCH / "sentinel_code", SCRATCH / "sentinel_data", SCRATCH / "sentinel_venv"
(DATA / "raw").mkdir(parents=True, exist_ok=True)


In [ ]:
# 2) Load the code bundle (Kaggle: attached dataset; Colab: upload when asked)
BUNDLE_NAME = "sentinel_gate_code.zip"

def find_bundle():
    zips = glob.glob(f"/kaggle/input/**/{BUNDLE_NAME}", recursive=True) + glob.glob(f"/content/{BUNDLE_NAME}")
    if zips:
        return Path(zips[0]), None
    dirs = [Path(p).parent for p in glob.glob("/kaggle/input/**/BUNDLE_INFO.json", recursive=True)]
    return None, (dirs[0] if dirs else None)

bundle, extracted = find_bundle()
if bundle is None and extracted is None and ON_COLAB:
    from google.colab import files
    files.upload()
    bundle, extracted = find_bundle()
if bundle is None and extracted is None:
    raise SystemExit(f"{BUNDLE_NAME} not found. Kaggle: Add Input -> your code-bundle dataset.")

shutil.rmtree(CODE, ignore_errors=True)
if bundle is not None:
    print(f"bundle: {bundle}\nsha256: {hashlib.sha256(bundle.read_bytes()).hexdigest()}")
    with zipfile.ZipFile(bundle) as zf:
        zf.extractall(CODE)
else:
    print(f"using extracted bundle at {extracted}")
    shutil.copytree(extracted, CODE)
for need in ("scripts/export_metadata_kaggle.py", "src/sentinel/backtest/metadata.py",
             f"docs/cleaning/{FILE_TYPE}.json"):
    if not (CODE / need).exists():
        raise SystemExit(f"Bundle has no {need}: it is an old bundle. Upload the Stage 7b dist/sentinel_gate_code.zip.")
print(open(CODE / "BUNDLE_INFO.json").read()[:200], "...")

In [ ]:
# 3) Install locked dependencies into an isolated Python 3.11 venv
def run(cmd, env=None, cwd=None):
    print("$", " ".join(map(str, cmd)), flush=True)
    proc = subprocess.Popen(list(map(str, cmd)), env=env, cwd=cwd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        if '"event": "clean_step"' not in line:
            print(line, end="", flush=True)
    if proc.wait() != 0:
        raise SystemExit(f"command failed with exit code {proc.returncode}")

run([sys.executable, "-m", "pip", "install", "--quiet", "uv==0.13.6"])
env = dict(os.environ, UV_PROJECT_ENVIRONMENT=str(VENV), UV_PYTHON="3.11", UV_LINK_MODE="copy")
run([sys.executable, "-m", "uv", "sync", "--frozen", "--no-dev", "--no-install-project"], env=env, cwd=CODE)
PY = VENV / "bin" / "python"
RUN_ENV = dict(os.environ, PYTHONPATH=f"{CODE / 'src'}:{CODE}", SENTINEL_DATA_DIR=str(DATA),
               HF_HOME=str(DATA / ".hf-home"), PYTHONUNBUFFERED="1")
run([PY, "-c", "import sys, thrember, lightgbm, sklearn; print(sys.version.split()[0], 'lightgbm', lightgbm.__version__, 'sklearn', sklearn.__version__)"], env=RUN_ENV)

In [ ]:
# 4) Download (SHA-256 verified) -> export metadata
t0 = time.time()
run([PY, "scripts/download_ember.py", "--file-types", FILE_TYPE, "--dest", DATA / "raw"], env=RUN_ENV, cwd=CODE)
t1 = time.time()
out_zip = OUT_DIR / f"sentinel_stage7b_meta_{FILE_TYPE}.zip"
run([PY, "scripts/export_metadata_kaggle.py", "--file-type", FILE_TYPE, "--raw-dir", DATA / "raw",
     "--out-zip", out_zip], env=RUN_ENV, cwd=CODE)
t2 = time.time()
print(f"== download {(t1 - t0) / 60:.1f} min, export {(t2 - t1) / 60:.1f} min")
print("result:", out_zip, f"{out_zip.stat().st_size / 1e6:.1f} MB",
      "sha256", hashlib.sha256(out_zip.read_bytes()).hexdigest())
if ON_COLAB:
    from google.colab import files
    files.download(str(out_zip))


In [ ]:
# 5) Summary (download the zip from the Output tab and give its path to Claude)
with zipfile.ZipFile(out_zip) as zf:
    print(sorted(zf.namelist()))
    info = json.loads(zf.read("metadata_info.json"))
print({k: v for k, v in info.items() if k != "raw_zip_sha256"})
print("Kaggle: download", out_zip.name, "from the Output tab.")
